# 01~06 코드 빈칸 채우기 정답

빈칸만 확인하지 말고 각 코드가 반환하는 객체의 타입과 shape도 함께 확인하세요.


## Set 1 정답

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('../../dataset/TV.csv')
df['spec_text'] = df[['Picture_quality', 'Speaker', 'Frequency']].agg(' '.join, axis=1)
df['refresh_rate'] = df['spec_text'].str.extract(
    r'(?P<refresh_rate>\d{2,3})\s*Hz', expand=False
)
df['high_quality'] = df['Picture_quality'].str.contains(r'4K|8K', na=False).astype(int)
df['review_ratio'] = df['Reviews'] / df['Ratings']
df['price_ratio'] = df['current_price'] / df['MRP']
model_df = df.dropna(subset=['review_ratio', 'price_ratio']).copy()
cols_X = ['review_ratio', 'MRP', 'price_ratio', 'high_quality']
X = model_df[cols_X]
y = model_df['Stars']
model = RandomForestRegressor(random_state=123)
model.fit(X, y)
importance = pd.Series(model.feature_importances_, index=cols_X)
display(importance.idxmax())


## Set 2 정답

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../dataset/galaxy_users.csv')
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
valid = df[service_cols].isin(['Yes', 'No']).all(axis=1)
base = df.loc[valid].copy()
base[service_cols] = base[service_cols].replace({'Yes': 1, 'No': 0}).astype(int)
base['service_count'] = base[service_cols].sum(axis=1)
base['used_month'] = base['TotalCharges'] // base['MonthlyCharges']
corr_abs = base[['tenure', 'MonthlyCharges', 'used_month']].corr().abs()
np.fill_diagonal(corr_abs.values, 0)
display(corr_abs.max().max())
y = base['Churn']


## Set 3 정답

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv('../../dataset/mobiles.csv')
upper = df['sales'].mean() + 2 * df['sales'].std()
focus = df.loc[df['sales'] > upper].copy()
X = pd.get_dummies(df.drop(columns='sales'), columns=['screen_size'], drop_first=False)
y = df['sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
rmse_by_k = {}
for k in [3, 5, 7, 9, 11]:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    rmse_by_k[k] = mean_squared_error(y_test, pred) ** 0.5
display(pd.Series(rmse_by_k).idxmin())


## Set 4 정답

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('../../dataset/sales_pos.csv')
df_user = df.groupby('user').agg(
    gender=('gender', 'first'), age_group=('age_group', 'first'),
    job=('job', 'first'), city=('city', 'first'), marital=('marital', 'first'),
    prod_count=('prod', 'nunique'), total_purchase=('purchase', 'sum')
)
df_user['gender'] = df_user['gender'].replace({'M': 1, 'F': 0})
df_user['age_group'] = pd.to_numeric(
    df_user['age_group'].str.extract(r'(\d+)', expand=False)
)
X = pd.get_dummies(df_user, columns=['job', 'city'])
X_scaled = MinMaxScaler().fit_transform(X)
model = KMeans(n_clusters=7, random_state=123, n_init=10)
labels = model.fit_predict(X_scaled)
score = silhouette_score(X_scaled, labels)
display(round(score, 2))


## Set 5 정답

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('../../dataset/card_cust.csv')
base = df.copy()
base['MINIMUM_PAYMENTS'] = base['MINIMUM_PAYMENTS'].fillna(base['MINIMUM_PAYMENTS'].mean())
corr_by_tenure = base.groupby('TENURE').apply(
    lambda group: group['BALANCE'].corr(group['CREDIT_LIMIT'])
)
X = base.drop(columns='CUST_ID').copy()
X_scaled = StandardScaler().fit_transform(X)
scores, labels_by_k = {}, {}
for k in range(2, 6):
    labels = KMeans(n_clusters=k, random_state=1234).fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
    labels_by_k[k] = labels
best_k = pd.Series(scores).idxmax()
X['cluster'] = labels_by_k[best_k]
train = base.loc[base['CUST_ID'] % 4 != 0].copy()
test = base.loc[base['CUST_ID'] % 4 == 0].copy()


## Set 6 정답

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('../../dataset/edu_enrollees.csv')
base = df.drop(columns=['city', 'company_size', 'company_type']).copy()
object_cols = base.select_dtypes(include='object').columns
base = base.dropna(subset=object_cols).copy()
base = base.loc[~base['experience'].isin(['>20', '<1'])].copy()
base['experience'] = base['experience'].astype(int)
dummy = pd.get_dummies(base['gender'], prefix='gender')
dummy = dummy.iloc[:, :-1].copy()
target_rate = base.groupby('relevant_experience')['target'].mean()
categorical_cols = [
    'gender', 'relevant_experience', 'enrolled_university',
    'education_level', 'major_discipline', 'last_new_job'
]
numeric_cols = ['city_development_index', 'experience', 'training_hours']
job2 = pd.get_dummies(
    base[numeric_cols + categorical_cols + ['target', 'Xgrp']],
    columns=categorical_cols,
    drop_first=True
)
X = job2.drop(columns=['target', 'Xgrp'])
y = job2['target']
model = LogisticRegression(C=100000, max_iter=1000, solver='liblinear', random_state=123)
model.fit(X, y)
odds_ratio = pd.Series(np.exp(model.coef_[0]), index=X.columns)
answer = np.floor(odds_ratio.max() * 100) / 100
train = job2.loc[job2['Xgrp'] == 'train'].copy()
test = job2.loc[job2['Xgrp'] == 'test'].copy()
